# Fractal geometries — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/fractals.html#exercises); this notebook
gives you a running start on each one, and the exercise data set is already
in its filesystem under `data/`.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, IntSlider
import chaosbook as cb

## The chaos game and the box-counting fit

The chapter fits the box counts of a Sierpinski gasket with a fixed number of
points. Here you choose the number of points and watch the fit: with too few
points the finest boxes each hold at most one point, the counts flatten and the
dimension is underestimated (the exact value is $\log 3/\log 2 = 1.585$).

In [ ]:
def boxcount_explorer(n=10000):
    X, Y = cb.chaos_game(n=n, seed=1)
    dim, boxsize, boxcount = cb.box_dimension(X, Y)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.6))
    ax1.plot(X, Y, "k.", markersize=1)
    ax1.set_aspect("equal")
    ax1.set_xlabel("$x$")
    ax1.set_ylabel("$y$")
    ax2.plot(np.log(boxsize), np.log(boxcount), "o", mfc="none", mec="k")
    slope, icpt = np.polyfit(np.log(boxsize), np.log(boxcount), 1)
    xx = np.log(boxsize[[0, -1]])
    ax2.plot(xx, slope * xx + icpt, "r")
    ax2.axhline(np.log(n), color="gray", ls=":")
    ax2.set_xlabel(r"$\log\varepsilon$")
    ax2.set_ylabel(r"$\log N(\varepsilon)$")
    ax2.set_title(f"$D_0 = {dim:.3f}$")
    fig.tight_layout()
    plt.show()

interact(boxcount_explorer, n=IntSlider(10000, min=200, max=20000, step=200));

## Exercise: correlation dimension

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html#exercises).
Below, a starting point for part a): a brute-force pair count. Replace it by
your own binned version if you like, and compare with `cb.correlation_dimension`.

In [ ]:
X, Y = cb.chaos_game(n=2000, seed=1)       # test set: Sierpinski gasket
P = np.column_stack([X, Y])

# a) the correlation integral C(r): fraction of pairs closer than r
d = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2))
d = d[np.triu_indices(len(P), k=1)]
r = np.logspace(-2, -1, 15)
C = np.array([(d < ri).mean() for ri in r])
D2 = np.polyfit(np.log(r), np.log(C), 1)[0]
print("D2 (own code)      =", D2)
print("cb.correlation_dimension =", cb.correlation_dimension(X, Y)[0])
plt.loglog(r, C, "o")
plt.xlabel("$r$")
plt.ylabel("$C(r)$")
plt.show()

In [ ]:
# b) the dripping-faucet attractor: the return plot (T_n, T_{n+1}) of the
# faucet map of the chapter on higher-order maps, then compare
# cb.correlation_dimension(x, y) with cb.box_dimension(x, y).
mc, delta, omega, mu = 1.0, 7.0, 60 * np.pi, 3.0

def yy(t, y0, v0):
    return np.exp(-delta * t) / omega * (
        (omega * np.cos(omega * t) + delta * np.sin(omega * t)) * y0
        + np.sin(omega * t) * v0)

def vv(t, y0, v0):
    return np.exp(-delta * t) / omega * (
        -(omega**2 + delta**2) * np.sin(omega * t) * y0
        + (omega * np.cos(omega * t) - delta * np.sin(omega * t)) * v0)

def faucet(s):
    T, m0, y0, v0 = s
    T = (mc - m0) / mu
    return np.array([T, mc / 2 - yy(T, y0, v0) / 9, 1.0, vv(T, y0, v0)])

S = cb.orbit(faucet, (0.0, mc / 2, 1.0, 0.0), 3000)
T = S[101:, 0]                               # drop the transient
x, y = T[:-1], T[1:]
plt.plot(x, y, "k.", markersize=2)
plt.xlabel("$T_n$"); plt.ylabel("$T_{n+1}$")
plt.show()
print("D2 =", cb.correlation_dimension(x, y)[0], "  D0 =", cb.box_dimension(x, y)[0])


## Exercise: fractal properties of the forced double-well oscillator

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html#exercises).
The system is autonomous in $(x, y, z)$ with $\dot z=\omega$, so sampling once
per forcing period $2\pi/\omega$ gives the Poincaré section of the figure.

In [ ]:
def double_well(t, s, A, f, omega):
    x, y, z = s
    return [y, -f * y + x - x**3 + A * np.cos(z), omega]

def poincare_cloud(A=0.3, f=0.15, omega=1.0, N=5000):
    T = 2 * np.pi / omega
    tt = T * np.arange(1, N + 1)
    sol = solve_ivp(double_well, (0, tt[-1]), [0, 0, 0], t_eval=tt,
                    args=(A, f, omega), rtol=1e-8, atol=1e-10)
    return sol.y[0][N // 2:], sol.y[1][N // 2:]        # drop the transient

# a) what does this do? why and when is it useful?
x, y = poincare_cloud(A=0.3)
plt.plot(x, y, "k.", markersize=2)
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.show()

In [ ]:
# b) the box dimension of this attractor
dim, boxsize, boxcount = cb.box_dimension(x, y)
print("D0 =", dim)

## Exercise: fractal curve

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html#exercises).
The points of the curve are in `data/FractalCurve.txt`.

In [ ]:
data = np.loadtxt("data/FractalCurve.txt")
x, y = data[:, 0], data[:, 1]
plt.plot(x, y, "k", linewidth=0.5)
plt.gca().set_aspect("equal")
plt.show()

# c) box-counting dimension of the curve
dim, boxsize, boxcount = cb.box_dimension(x, y)
print("D0 =", dim)

In [ ]:
# a)-b): express d in terms of phi, then count the recursive steps until the
# length exceeds 1 km. Each step multiplies the length by a factor you can
# read off from the generator; start from phi = pi/7.
phi = np.pi / 7

## Exercise: multi-fractals in threes

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html#exercises).
The binomial construction of the chapter generalizes to three branches by
repeating the weight vector `[p1, p2, p3]` through a Kronecker product.

In [ ]:
# a) N = 3^6 points: the measure after k = 6 steps
p = np.array([0.8, 0.0, 0.2])           # try other weights, e.g. [1/2, 0, 1/2]
k = 6
mass = np.array([1.0])
for _ in range(k):
    mass = np.kron(mass, p)
x = (np.arange(3**k) + 0.5) / 3**k
plt.vlines(x, 0, mass, color="k", linewidth=0.8)
plt.xlabel("position")
plt.ylabel("$p_i$")
plt.show()

In [ ]:
# d) D(q) by box counting, q in [-10, 10] without q = 1. cb.generalized_dimensions
# halves the boxes each time; here the boxes must be divided in three, so write
# the loop yourself: Z(q, eps) = sum of mass**q over occupied boxes at eps = 3**-j.
qs = [q for q in range(-10, 11) if q != 1]
# Your turn: for each q fit log Z against log eps and divide the slope by (q - 1)
# (hint: reshape(mass, (3**j, -1)).sum(axis=1) gives the box masses at level j;
# keep only boxes with mass > 0). Compare with the analytical result of c).

---
*Back to [the chapter](https://mvreeuwijk.github.io/chaosbook/fractals.html) — or open the
[cookbook](cookbook.ipynb) in this same lab environment.*